# ICU Patient Phenotypes from MIMIC-III: Vitals vs. Comorbidities

Doctors already group patients into rough categories in their heads - "the sick ones," "the stable ones." This notebook tries to do that in a more systematic way, using two different views of the same patients, and checks whether the two views agree.

If any of these terms are new, they're explained the first time they come up:

- **Clustering**: grouping data points (here, patients) so that similar ones end up in the same group, without being told in advance what the groups should be. This is different from the readmission-prediction style of model, which learns from a known label - clustering has no label, it just finds structure.
- **Feature**: a piece of information about a patient that we give to the model.
- **Cluster**: one of the groups the model finds.

We build two separate feature sets for the same ICU patients, cluster each one, and compare the results:

- **Vitals/labs features**: how sick the patient looks right now (first 24 hours of their ICU stay).
- **Comorbidity features**: what chronic conditions the patient came in with, based on their diagnosis codes.

Data source: MIMIC-III, accessed through Google BigQuery (`physionet-data.mimiciii_clinical`).


## 1. Install the libraries we need

In [ ]:
# pandas/numpy handle the data, scikit-learn does the clustering and scaling,
# scipy runs the statistical tests, matplotlib/seaborn make the plots.
!pip -q install --upgrade scikit-learn scipy matplotlib seaborn pandas umap-learn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from scipy import stats

try:
    import umap
    HAVE_UMAP = True
except ImportError:
    HAVE_UMAP = False

# fixed seed so the clustering comes out the same way every time we run it
SEED = 42
np.random.seed(SEED)
sns.set_style("whitegrid")


## 2. Connect to BigQuery

MIMIC-III is hosted on BigQuery. This logs in and opens a connection so we can query it directly from the notebook.

In [ ]:
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

PROJECT_ID = 'mc-ut-msai-aih-1'  # change this to your own BigQuery project ID
client = bigquery.Client(project=PROJECT_ID)

test_query = "SELECT COUNT(*) AS total FROM `physionet-data.mimiciii_clinical.icustays`"
print(client.query(test_query).to_dataframe())


## 3. Build the cohort

One row in our table = one ICU stay. We keep things simple by using each patient's **first** ICU stay only - if we kept every stay, a patient who came back five times would count five times and skew the clusters toward whatever that one patient looked like.

Other cohort rules:
- Adults only (age 18+ at ICU admission).
- ICU stay of at least 24 hours, since we need a full first-24h window to build features.
- Newborn admissions excluded (same reasoning as the readmission tutorial - they behave very differently from adult stays).

One MIMIC-specific detail: to protect patient privacy, MIMIC-III shifts the recorded age of anyone over 89, which can make some patients appear to be 200+ years old. We cap age at 90 below.


In [ ]:
cohort_query = """
WITH icu_ranked AS (
  SELECT
    i.subject_id, i.hadm_id, i.icustay_id, i.intime, i.outtime,
    TIMESTAMP_DIFF(i.outtime, i.intime, HOUR) AS los_hours,
    ROW_NUMBER() OVER (PARTITION BY i.subject_id ORDER BY i.intime) AS stay_rank
  FROM `physionet-data.mimiciii_clinical.icustays` i
)
SELECT
  r.subject_id, r.hadm_id, r.icustay_id, r.intime, r.los_hours,
  LEAST(DATE_DIFF(DATE(r.intime), DATE(p.dob), YEAR), 90) AS age,
  p.gender,
  a.hospital_expire_flag,
  a.admission_type
FROM icu_ranked r
JOIN `physionet-data.mimiciii_clinical.patients` p ON r.subject_id = p.subject_id
JOIN `physionet-data.mimiciii_clinical.admissions` a ON r.hadm_id = a.hadm_id
WHERE r.stay_rank = 1
  AND r.los_hours >= 24
  AND a.admission_type != 'NEWBORN'
  AND DATE_DIFF(DATE(r.intime), DATE(p.dob), YEAR) >= 18
"""

cohort = client.query(cohort_query).to_dataframe()
print(f"Cohort size: {len(cohort):,} ICU stays")
cohort.head()


## 4. Feature set 1: vitals and labs (first 24 hours)

A vital sign like heart rate is measured many times over the first 24 hours, not just once, so we can't use the raw readings directly. We collapse each vital into a few summary numbers instead: the **mean** (typical value), **min** and **max** (the extremes), and **std** (how much it fluctuated). The extremes and fluctuation matter clinically - a patient whose heart rate spiked to 160 at some point looks different from one who stayed steady, even if their average is the same.

We do this aggregation directly in BigQuery rather than pulling raw `chartevents` rows into the notebook, since `chartevents` is enormous (hundreds of millions of rows).


In [ ]:
VITAL_ITEMIDS = {
    "heart_rate": [211, 220045],
    "resp_rate":  [618, 615, 220210, 224690],
    "sbp":        [51, 442, 455, 6701, 220179, 220050],
    "dbp":        [8368, 8440, 8441, 8555, 220180, 220051],
    "temp_c":     [676, 223762],
    "spo2":       [646, 220277],
    "gcs":        [198, 226755],
}

# build one UNION ALL query that tags each row with a readable vital name
vital_cases = "\n  ".join(
    f"WHEN itemid IN ({','.join(map(str, ids))}) THEN '{name}'"
    for name, ids in VITAL_ITEMIDS.items()
)
all_vital_ids = sorted({i for ids in VITAL_ITEMIDS.values() for i in ids})

vitals_query = f"""
WITH tagged AS (
  SELECT
    c.icustay_id,
    CASE {vital_cases} END AS vital,
    c.valuenum
  FROM `physionet-data.mimiciii_clinical.chartevents` c
  JOIN `physionet-data.mimiciii_clinical.icustays` i ON c.icustay_id = i.icustay_id
  WHERE c.itemid IN ({','.join(map(str, all_vital_ids))})
    AND c.valuenum IS NOT NULL
    AND c.charttime BETWEEN i.intime AND TIMESTAMP_ADD(i.intime, INTERVAL 24 HOUR)
)
SELECT
  icustay_id, vital,
  AVG(valuenum) AS mean_val,
  MIN(valuenum) AS min_val,
  MAX(valuenum) AS max_val,
  STDDEV(valuenum) AS std_val
FROM tagged
WHERE vital IS NOT NULL
GROUP BY icustay_id, vital
"""

vitals_long = client.query(vitals_query).to_dataframe()
print(f"Rows: {len(vitals_long):,}")
vitals_long.head()


In [ ]:
# reshape from one-row-per-(icustay, vital) to one-row-per-icustay,
# with columns like heart_rate_mean, heart_rate_min, ...
vitals_wide = vitals_long.pivot(index="icustay_id", columns="vital",
                                 values=["mean_val", "min_val", "max_val", "std_val"])
vitals_wide.columns = [f"{v}_{stat.replace('_val','')}" for stat, v in vitals_wide.columns]
vitals_wide = vitals_wide.reset_index()

print(f"Vitals feature table: {vitals_wide.shape}")
vitals_wide.head()


We do the same thing for a handful of labs that reflect acute severity: creatinine (kidney function), white blood cell count, lactate (tissue oxygenation), glucose, bicarbonate, potassium, and hemoglobin.

In [ ]:
LAB_ITEMIDS = {
    "creatinine":  [50912],
    "wbc":         [51301],
    "lactate":     [50813],
    "glucose":     [50931],
    "bicarbonate": [50882],
    "potassium":   [50971],
    "hemoglobin":  [51222],
}
lab_cases = "\n  ".join(
    f"WHEN itemid IN ({','.join(map(str, ids))}) THEN '{name}'"
    for name, ids in LAB_ITEMIDS.items()
)
all_lab_ids = sorted({i for ids in LAB_ITEMIDS.values() for i in ids})

labs_query = f"""
WITH tagged AS (
  SELECT
    l.hadm_id,
    CASE {lab_cases} END AS lab,
    l.valuenum
  FROM `physionet-data.mimiciii_clinical.labevents` l
  JOIN `physionet-data.mimiciii_clinical.icustays` i ON l.hadm_id = i.hadm_id
  WHERE l.itemid IN ({','.join(map(str, all_lab_ids))})
    AND l.valuenum IS NOT NULL
    AND l.charttime BETWEEN i.intime AND TIMESTAMP_ADD(i.intime, INTERVAL 24 HOUR)
)
SELECT hadm_id, lab, AVG(valuenum) AS mean_val, MIN(valuenum) AS min_val, MAX(valuenum) AS max_val
FROM tagged
WHERE lab IS NOT NULL
GROUP BY hadm_id, lab
"""

labs_long = client.query(labs_query).to_dataframe()
labs_wide = labs_long.pivot(index="hadm_id", columns="lab", values=["mean_val", "min_val", "max_val"])
labs_wide.columns = [f"{l}_{stat.replace('_val','')}" for stat, l in labs_wide.columns]
labs_wide = labs_wide.reset_index()

print(f"Labs feature table: {labs_wide.shape}")
labs_wide.head()


## 5. Feature set 2: comorbidities from diagnosis codes

A **comorbidity** is a chronic condition a patient has alongside whatever brought them into the ICU this time - diabetes, heart failure, kidney disease, and so on. We pull these from `diagnoses_icd`, which lists every ICD-9 code billed for the admission, not just the main reason for the visit.

We map ICD-9 codes to a simplified set of comorbidity categories (a subset of the standard Elixhauser index) using the code prefixes below. Each category becomes a **binary feature**: 1 if the patient has any code in that category, 0 if not. This is a coarser mapping than the full 30-category Elixhauser index, but it's enough to capture the main chronic-disease groupings and keeps the SQL readable.


In [ ]:
COMORBIDITY_PREFIXES = {
    "congestive_heart_failure": ["428"],
    "cardiac_arrhythmia":       ["4273", "4270", "4272"],
    "hypertension":             ["401"],
    "chronic_pulmonary_disease":["490","491","492","493","494","495","496"],
    "diabetes":                 ["250"],
    "renal_failure":            ["585","586"],
    "liver_disease":            ["570","571","572"],
    "metastatic_cancer":        ["196","197","198","199"],
    "solid_tumor":              ["140","141","142","143","144","145","146","147","148","149",
                                  "150","151","152","153","154","155","156","157","158","159"],
    "coagulopathy":             ["286"],
    "obesity":                  ["2780"],
    "fluid_electrolyte":        ["2760","2761","2762","2763","2764","2765","2766","2767","2768","2769"],
    "anemia":                   ["280","281","282","283","284","285"],
    "alcohol_abuse":            ["3050","291"],
    "depression":               ["2962","2963","311"],
}

# build one CASE WHEN per category, matching on ICD9 code prefix with STARTS_WITH
case_lines = []
for name, prefixes in COMORBIDITY_PREFIXES.items():
    conditions = " OR ".join(f"STARTS_WITH(icd9_code, '{p}')" for p in prefixes)
    case_lines.append(f"MAX(CASE WHEN {conditions} THEN 1 ELSE 0 END) AS {name}")
case_sql = ",\n  ".join(case_lines)

comorbidity_query = f"""
SELECT
  hadm_id,
  {case_sql}
FROM `physionet-data.mimiciii_clinical.diagnoses_icd`
GROUP BY hadm_id
"""

comorbidities = client.query(comorbidity_query).to_dataframe()
print(f"Comorbidity feature table: {comorbidities.shape}")
comorbidities.head()


## 6. Put it together, and look at the data before modeling

Same rule as before: look at the outcome and the missingness before doing anything else, so we know what we're working with.


In [ ]:
features = cohort.merge(vitals_wide, on="icustay_id", how="left")
features = features.merge(labs_wide, on="hadm_id", how="left")
features = features.merge(comorbidities, on="hadm_id", how="left")

vital_lab_cols = [c for c in list(vitals_wide.columns) + list(labs_wide.columns)
                  if c not in ("icustay_id", "hadm_id")]
comorbidity_cols = [c for c in comorbidities.columns if c != "hadm_id"]

print(f"Combined table: {features.shape}")
print(f"Vitals/labs columns: {len(vital_lab_cols)}")
print(f"Comorbidity columns: {len(comorbidity_cols)}")


In [ ]:
print(features["hospital_expire_flag"].value_counts(normalize=True).round(3))

sns.countplot(data=features, x="hospital_expire_flag")
plt.title("In-hospital mortality in this cohort")
plt.xlabel("Died in hospital")
plt.ylabel("Number of ICU stays")
plt.show()


In [ ]:
# drop stays missing more than half of their vitals/labs features - a small number,
# safe to remove rather than impute over that much missing data
missing_frac = features[vital_lab_cols].isna().mean(axis=1)
features = features[missing_frac <= 0.5].reset_index(drop=True)

# comorbidity flags should never be missing (GROUP BY guarantees a row per hadm_id
# that has any diagnosis code), but fill any remaining gaps with 0 just in case
features[comorbidity_cols] = features[comorbidity_cols].fillna(0)

print(f"Cohort after missingness filter: {len(features):,}")


## 7. Get the vitals/labs ready for clustering

The comorbidity features are already 0/1, so they don't need scaling. The vitals/labs features are continuous and on very different scales (heart rate in the 60-140 range, lactate in the 0-10 range), so we impute missing values and standardize them the same way as before - otherwise a feature with naturally large numbers would dominate the distance calculation clustering relies on, just because of its size.


In [ ]:
imputer = SimpleImputer(strategy="median")
X_vitals_imputed = imputer.fit_transform(features[vital_lab_cols])

scaler = StandardScaler()
X_vitals = scaler.fit_transform(X_vitals_imputed)
X_vitals = pd.DataFrame(X_vitals, columns=vital_lab_cols)

X_comorbid = features[comorbidity_cols].astype(int)

print(f"Vitals/labs matrix: {X_vitals.shape}")
print(f"Comorbidity matrix: {X_comorbid.shape}")


## 8. Cluster on vitals/labs

**K-Means** groups patients into *k* clusters by repeatedly assigning each patient to the nearest cluster center and then recomputing the centers, until the assignments stop changing. It needs *k* chosen in advance, so we try a range of values and use the **silhouette score** to pick one - this score is high when patients are close to others in their own cluster and far from patients in other clusters, and low otherwise.


In [ ]:
k_range = range(2, 8)
silhouettes_vitals = []
for k in k_range:
    labels = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit_predict(X_vitals)
    silhouettes_vitals.append(silhouette_score(X_vitals, labels))

plt.plot(list(k_range), silhouettes_vitals, marker="o")
plt.title("Silhouette score by k - vitals/labs clustering")
plt.xlabel("k"); plt.ylabel("Silhouette score")
plt.show()

k_vitals = list(k_range)[int(np.argmax(silhouettes_vitals))]
print(f"Chosen k for vitals clustering: {k_vitals}")


In [ ]:
kmeans_vitals = KMeans(n_clusters=k_vitals, random_state=SEED, n_init=10)
features["cluster_vitals"] = kmeans_vitals.fit_predict(X_vitals)

print(features["cluster_vitals"].value_counts().sort_index())


## 9. Cluster on comorbidities

Same method, applied to the comorbidity flags instead. We pick *k* the same way.


In [ ]:
silhouettes_comorbid = []
for k in k_range:
    labels = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit_predict(X_comorbid)
    silhouettes_comorbid.append(silhouette_score(X_comorbid, labels))

plt.plot(list(k_range), silhouettes_comorbid, marker="o", color="darkorange")
plt.title("Silhouette score by k - comorbidity clustering")
plt.xlabel("k"); plt.ylabel("Silhouette score")
plt.show()

k_comorbid = list(k_range)[int(np.argmax(silhouettes_comorbid))]
print(f"Chosen k for comorbidity clustering: {k_comorbid}")


In [ ]:
kmeans_comorbid = KMeans(n_clusters=k_comorbid, random_state=SEED, n_init=10)
features["cluster_comorbid"] = kmeans_comorbid.fit_predict(X_comorbid)

print(features["cluster_comorbid"].value_counts().sort_index())


## 10. Do the two clusterings agree?

We now have two cluster labels per patient - one from vitals, one from comorbidities. If they mostly agree, that would suggest a patient's acute severity and chronic disease burden move together. If they mostly disagree, that's evidence they're capturing genuinely different things.

Two standard scores for comparing two sets of cluster labels:

- **Adjusted Rand Index (ARI)**: how often the two clusterings agree on which pairs of patients belong together, corrected for chance agreement. 1.0 means identical clusterings, 0.0 means no better than random.
- **Normalized Mutual Information (NMI)**: how much knowing one clustering tells you about the other, on a 0-1 scale.


In [ ]:
ari = adjusted_rand_score(features["cluster_vitals"], features["cluster_comorbid"])
nmi = normalized_mutual_info_score(features["cluster_vitals"], features["cluster_comorbid"])

print(f"Adjusted Rand Index: {ari:.3f}")
print(f"Normalized Mutual Information: {nmi:.3f}")

pd.crosstab(features["cluster_vitals"], features["cluster_comorbid"])


A low ARI/NMI here would mean a patient can look high-risk on one axis (say, a rough comorbidity cluster) but unremarkable on the other (a calm vitals cluster right now) - which is a real clinical possibility, not a modeling failure. We check this against actual outcomes next.

## 11. Do the clusters mean anything clinically?

Neither clustering used mortality or length of stay as an input. Here we check, after the fact, whether the clusters we found actually differ on those outcomes - evidence the clustering found real structure rather than noise.


In [ ]:
def profile_clusters(df, cluster_col, outcome_cols=("hospital_expire_flag", "los_hours")):
    summary = df.groupby(cluster_col).agg(
        n=("icustay_id", "count"),
        mortality_rate=("hospital_expire_flag", "mean"),
        los_mean_hours=("los_hours", "mean"),
    )
    return summary.round(3)

print("Vitals clusters:")
print(profile_clusters(features, "cluster_vitals"))
print("\nComorbidity clusters:")
print(profile_clusters(features, "cluster_comorbid"))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(data=features, x="cluster_vitals", y="hospital_expire_flag", ax=axes[0], errorbar=None)
axes[0].set_title("Mortality rate by vitals cluster")
axes[0].set_ylabel("Mortality rate")

sns.barplot(data=features, x="cluster_comorbid", y="hospital_expire_flag", ax=axes[1], errorbar=None)
axes[1].set_title("Mortality rate by comorbidity cluster")
axes[1].set_ylabel("Mortality rate")

plt.tight_layout()
plt.show()


In [ ]:
# chi-square test: does mortality rate differ significantly across clusters?
for label, col in [("vitals", "cluster_vitals"), ("comorbidity", "cluster_comorbid")]:
    contingency = pd.crosstab(features[col], features["hospital_expire_flag"])
    chi2, p_val, _, _ = stats.chi2_contingency(contingency)
    print(f"{label} clusters vs. mortality: chi2={chi2:.2f}, p={p_val:.4g}")

# one-way ANOVA: does length of stay differ significantly across clusters?
for label, col in [("vitals", "cluster_vitals"), ("comorbidity", "cluster_comorbid")]:
    groups = [g["los_hours"].values for _, g in features.groupby(col)]
    f_stat, p_val = stats.f_oneway(*groups)
    print(f"{label} clusters vs. LOS: F={f_stat:.2f}, p={p_val:.4g}")


To see what each vitals cluster actually looks like physiologically, we can compare each cluster's average feature values against the overall population average - this is how an unlabeled cluster number gets turned into a description like "high-lactate, low-GCS."


In [ ]:
vitals_profile = features.groupby("cluster_vitals")[vital_lab_cols].mean().T
vitals_profile["overall_mean"] = features[vital_lab_cols].mean()
vitals_profile.round(2)


In [ ]:
comorbid_profile = features.groupby("cluster_comorbid")[comorbidity_cols].mean().T
comorbid_profile["overall_rate"] = features[comorbidity_cols].mean()
comorbid_profile.round(2)


## 12. What we found

*(Fill this in once you've run the notebook on the real cohort - this section is what your presentation's evaluation slide should summarize.)*

Questions to answer from the output above:

- How many clusters did each feature set produce, and how big is each one?
- Looking at `vitals_profile`, what physiologically distinguishes each vitals cluster from the population average? Does one look like a shock/sepsis-like phenotype, another like a lower-acuity group?
- Looking at `comorbid_profile`, what chronic-disease pattern distinguishes each comorbidity cluster?
- What were the ARI and NMI values? Do the two clusterings mostly agree, mostly disagree, or partially overlap? What would that mean clinically - does a patient's acute state track their chronic burden, or are they largely independent?
- Were the chi-square (mortality) and ANOVA (LOS) tests significant for each clustering? Which feature set separated outcomes better?

## Limitations

- The comorbidity mapping here is a simplified subset of Elixhauser (15 categories via ICD-9 prefix matching), not the full validated index - a real deployment would use a maintained mapping table.
- Vitals/labs features only use the first 24 hours; a patient who crashes on day 3 looks the same as one who stayed stable, at least in this feature set.
- We didn't tune anything beyond *k* - no comparison against other clustering algorithms (e.g. Gaussian Mixture Models, hierarchical clustering), and no hyperparameter search.
- Cluster count and content can shift a bit if re-run with a different `k_range` or a different missingness cutoff - the qualitative pattern (whether the two clusterings agree) is the important, more stable result to report, not the exact ARI value.

## How to reproduce this

1. Get PhysioNet-credentialed access to MIMIC-III, and load it into Google BigQuery.
2. Run the SQL queries in Sections 3-5 to build the cohort, vitals/labs, and comorbidity tables.
3. Install scikit-learn and scipy.
4. Run Sections 6 through 11 in order - no manual steps needed in between.
